In [1]:
# Q5 - Contact channel and calendar

# Question: does contact method, month and weekday change the yes rate?

# Each group is compared with the overall yes rate (the Q1 baseline) and shown with its customer count. Groups under 100 customers are flagged and not interpreted. The data has no year column and no January or February, so a month combines several years.

In [2]:
import pandas as pd

CLEAN_PATH = "../../data/bank_marketing_clean.parquet"
clean_df = pd.read_parquet(CLEAN_PATH)

clean_df.shape

(41176, 22)

In [3]:
# Q5, step 1: yes rate by contact type, month and weekday
subscribed = clean_df["subscribed_term_deposit"] == "yes"
baseline_rate = subscribed.mean() * 100

group_orders = {
    "contact_type": ["cellular", "telephone"],
    "contact_month": ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"],
    "contact_weekday": ["mon", "tue", "wed", "thu", "fri"],
}

summaries = []
for column, order in group_orders.items():
    summary = (
        pd.DataFrame({"value": clean_df[column], "subscribed": subscribed})
        .groupby("value")["subscribed"]
        .agg(customers="count", subscribers="sum")
        .reindex(order)
    )
    summary["yes_rate_pct"] = (summary["subscribers"] / summary["customers"] * 100).round(2)
    summary["vs_baseline_points"] = (summary["yes_rate_pct"] - baseline_rate).round(2)
    summary["share_of_customers_pct"] = (summary["customers"] / len(clean_df) * 100).round(2)
    summary["share_of_subscribers_pct"] = (summary["subscribers"] / subscribed.sum() * 100).round(2)
    summary["small_group"] = summary["customers"] < 100
    summaries.append(summary)

    print(f"{column}: customers {summary['customers'].sum()} (should be {len(clean_df)}), "
          f"subscribers {summary['subscribers'].sum()} (should be {subscribed.sum()})")

print(f"Baseline yes rate: {baseline_rate:.2f}%")
print()
pd.concat(summaries, keys=list(group_orders), names=["column", "value"])

contact_type: customers 41176 (should be 41176), subscribers 4639 (should be 4639)
contact_month: customers 41176 (should be 41176), subscribers 4639 (should be 4639)
contact_weekday: customers 41176 (should be 41176), subscribers 4639 (should be 4639)
Baseline yes rate: 11.27%



customers  subscribers  yes_rate_pct  \
column          value                                             
contact_type    cellular       26135         3852         14.74   
                telephone      15041          787          5.23   
contact_month   mar              546          276         50.55   
                apr             2631          539         20.49   
                may            13767          886          6.44   
                jun             5318          559         10.51   
                jul             7169          648          9.04   
                aug             6176          655         10.61   
                sep              570          256         44.91   
                oct              717          315         43.93   
                nov             4100          416         10.15   
                dec              182           89         48.90   
contact_weekday mon             8512          847          9.95   
                tue             8086          953         11.79   
                wed             8134          949         11.67   
                thu             8618         1044         12.11   
                fri             7826          846         10.81   

                           vs_baseline_points  share_of_customers_pct  \
column          value                                                   
contact_type    cellular                 3.47                   63.47   
                telephone               -6.04                   36.53   
contact_month   mar                     39.28                    1.33   
                apr                      9.22                    6.39   
                may                     -4.83                   33.43   
                jun                     -0.76                   12.92   
                jul                     -2.23                   17.41   
                aug                     -0.66                   15.00   
                sep                     33.64                    1.38   
                oct                     32.66                    1.74   
                nov                     -1.12                    9.96   
                dec                     37.63                    0.44   
contact_weekday mon                     -1.32                   20.67   
                tue                      0.52                   19.64   
                wed                      0.40                   19.75   
                thu                      0.84                   20.93   
                fri                     -0.46                   19.01   

                           share_of_subscribers_pct  small_group  
column          value                                             
contact_type    cellular                      83.04        False  
                telephone                     16.96        False  
contact_month   mar                            5.95        False  
                apr                           11.62        False  
                may                           19.10        False  
                jun                           12.05        False  
                jul                           13.97        False  
                aug                           14.12        False  
                sep                            5.52        False  
                oct                            6.79        False  
                nov                            8.97        False  
                dec                            1.92        False  
contact_weekday mon                           18.26        False  
                tue                           20.54        False  
                wed                           20.46        False  
                thu                           22.50        False  
                fri                           18.24        False

In [4]:
# Q5, step 2: contact type against month
month_order = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
type_order = ["cellular", "telephone"]

frame = pd.DataFrame({
    "month": clean_df["contact_month"],
    "contact_type": clean_df["contact_type"],
    "subscribed": subscribed,
})

grouped = frame.groupby(["month", "contact_type"])["subscribed"].agg(customers="count", subscribers="sum")
customers = (grouped["customers"].unstack("contact_type")
             .reindex(index=month_order, columns=type_order).fillna(0).astype(int))
subscribers = (grouped["subscribers"].unstack("contact_type")
               .reindex(index=month_order, columns=type_order).fillna(0).astype(int))

print(f"Customers: {customers.values.sum()} (should be {len(clean_df)}), "
      f"subscribers: {subscribers.values.sum()} (should be {subscribed.sum()})")
print(f"Cellular: {customers['cellular'].sum()} customers / {subscribers['cellular'].sum()} subscribers (should be 26135 / 3852)")
print(f"Telephone: {customers['telephone'].sum()} customers / {subscribers['telephone'].sum()} subscribers (should be 15041 / 787)")
print()

# Inside each month: how does the two channels compare?
channel_rates = (subscribers / customers * 100).where(customers >= 100).round(2)
summary = pd.DataFrame({
    "cellular_customers": customers["cellular"],
    "telephone_customers": customers["telephone"],
    "telephone_share_pct": (customers["telephone"] / customers.sum(axis=1) * 100).round(1),
    "cellular_yes_rate_pct": channel_rates["cellular"],
    "telephone_yes_rate_pct": channel_rates["telephone"],
})
summary["gap_points"] = (summary["cellular_yes_rate_pct"] - summary["telephone_yes_rate_pct"]).round(2)
summary["month_yes_rate_pct"] = (subscribers.sum(axis=1) / customers.sum(axis=1) * 100).round(2)

print("By month and contact type (channel yes rates blank where under 100 customers):")
print(summary)
print()

# Does each month still stand out once its contact mix is allowed for?
rows = []
for month in month_order:
    in_month = frame["month"] == month
    other_rates = frame[~in_month].groupby("contact_type")["subscribed"].mean().reindex(type_order)
    month_counts = customers.loc[month]

    assert not ((month_counts > 0) & other_rates.isna()).any(), f"no comparison customers for {month}"
    expected_subscribers = (month_counts * other_rates).sum()

    month_customers = month_counts.sum()
    observed_subscribers = subscribers.loc[month].sum()
    observed_rate = observed_subscribers / month_customers * 100
    expected_rate = expected_subscribers / month_customers * 100
    margin = 1.96 * (expected_rate / 100 * (1 - expected_rate / 100) / month_customers) ** 0.5 * 100

    rows.append({
        "month": month,
        "customers": month_customers,
        "observed_yes_rate_pct": round(observed_rate, 2),
        "expected_yes_rate_pct": round(expected_rate, 2),
        "difference_points": round(observed_rate - expected_rate, 2),
        "chance_margin_points": round(margin, 2),
    })

month_vs_expected = pd.DataFrame(rows).set_index("month")
month_vs_expected["outside_chance_margin"] = (
    month_vs_expected["difference_points"].abs() > month_vs_expected["chance_margin_points"]
)

print("Each month: actual yes rate vs the rate expected from its contact mix:")
month_vs_expected

Customers: 41176 (should be 41176), subscribers: 4639 (should be 4639)
Cellular: 26135 customers / 3852 subscribers (should be 26135 / 3852)
Telephone: 15041 customers / 787 subscribers (should be 15041 / 787)

By month and contact type (channel yes rates blank where under 100 customers):
       cellular_customers  telephone_customers  telephone_share_pct  \
month                                                                 
mar                   486                   60                 11.0   
apr                  2444                  187                  7.1   
may                  5517                 8250                 59.9   
jun                   820                 4498                 84.6   
jul                  6092                 1077                 15.0   
aug                  5907                  269                  4.4   
sep                   482                   88                 15.4   
oct                   563                  154                 21.5   


,customers,observed_yes_rate_pct,expected_yes_rate_pct,difference_points,chance_margin_points,outside_chance_margin
month,,,,,,
mar,546,50.55,13.05,37.50,2.83,True
apr,2631,20.49,13.53,6.96,1.31,True
may,13767,6.44,10.84,-4.40,0.52,True
jun,5318,10.51,6.76,3.75,0.67,True
jul,7169,9.04,14.61,-5.58,0.82,True
aug,6176,10.61,15.51,-4.90,0.90,True
sep,570,44.91,12.71,32.21,2.73,True
oct,717,43.93,12.10,31.84,2.39,True
nov,4100,10.15,14.43,-4.29,1.08,True
